In [1]:
import gzip
import json
from pathlib import Path
from collections import Counter

import pandas as pd
from tqdm import tqdm

# repo root - adjust if needed
DATA_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# verify the files we have
print("Files in data/raw:")
for f in sorted(DATA_DIR.glob("*.json.gz")):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name}: {size_mb:.1f} MB")

Files in data/raw:
  goodreads_book_authors.json.gz: 17.2 MB
  goodreads_book_genres_initial.json.gz: 23.1 MB
  goodreads_books_fantasy_paranormal.json.gz: 265.9 MB
  goodreads_books_history_biography.json.gz: 309.5 MB
  goodreads_books_mystery_thriller_crime.json.gz: 220.6 MB


In [2]:
# read just the first record from the fantasy file
fantasy_file = DATA_DIR / "goodreads_books_fantasy_paranormal.json.gz"

with gzip.open(fantasy_file, "rt", encoding="utf-8") as f:
    first_line = f.readline()
    sample = json.loads(first_line)

# pretty-print the structure
print(json.dumps(sample, indent=2)[:3000])  # first 3000 chars
print("\n\nAll keys:")
print(list(sample.keys()))

{
  "isbn": "",
  "text_reviews_count": "7",
  "series": [
    "189911"
  ],
  "country_code": "US",
  "language_code": "eng",
  "popular_shelves": [
    {
      "count": "58",
      "name": "to-read"
    },
    {
      "count": "15",
      "name": "fantasy"
    },
    {
      "count": "6",
      "name": "fiction"
    },
    {
      "count": "5",
      "name": "owned"
    },
    {
      "count": "3",
      "name": "hardcover"
    },
    {
      "count": "2",
      "name": "shelfari-favorites"
    },
    {
      "count": "2",
      "name": "series"
    },
    {
      "count": "1",
      "name": "might-read"
    },
    {
      "count": "1",
      "name": "dnf-d"
    },
    {
      "count": "1",
      "name": "hambly-barbara"
    },
    {
      "count": "1",
      "name": "strong-females"
    },
    {
      "count": "1",
      "name": "first-in-series"
    },
    {
      "count": "1",
      "name": "fantasy-sword-sorcery"
    },
    {
      "count": "1",
      "name": "no-thanks-series-co

In [3]:
def count_lines(filepath):
    count = 0
    with gzip.open(filepath, "rt", encoding="utf-8") as f:
        for _ in f:
            count += 1
    return count

total = count_lines(fantasy_file)
print(f"Total fantasy books: {total:,}")

Total fantasy books: 258,585


In [4]:
# stream through and audit description field
description_stats = {
    "total": 0,
    "missing": 0,
    "empty_string": 0,
    "very_short": 0,  # < 50 chars
    "short": 0,       # 50-100 chars
    "decent": 0,      # 100-500 chars
    "long": 0,        # 500+ chars
}

sample_descriptions = {"very_short": [], "decent": [], "long": []}

with gzip.open(fantasy_file, "rt", encoding="utf-8") as f:
    for line in tqdm(f, total=258585):
        book = json.loads(line)
        description_stats["total"] += 1
        
        desc = book.get("description", "")
        if desc is None:
            description_stats["missing"] += 1
        elif desc == "":
            description_stats["empty_string"] += 1
        elif len(desc) < 50:
            description_stats["very_short"] += 1
            if len(sample_descriptions["very_short"]) < 3:
                sample_descriptions["very_short"].append((book.get("title", ""), desc))
        elif len(desc) < 100:
            description_stats["short"] += 1
        elif len(desc) < 500:
            description_stats["decent"] += 1
            if len(sample_descriptions["decent"]) < 2:
                sample_descriptions["decent"].append((book.get("title", ""), desc))
        else:
            description_stats["long"] += 1
            if len(sample_descriptions["long"]) < 2:
                sample_descriptions["long"].append((book.get("title", ""), desc[:300] + "..."))

print("\nDescription length distribution:")
for k, v in description_stats.items():
    pct = 100 * v / description_stats["total"] if k != "total" else 100
    print(f"  {k:15s}: {v:>8,} ({pct:.1f}%)")

print("\n=== Sample 'very_short' descriptions (probably garbage) ===")
for title, desc in sample_descriptions["very_short"]:
    print(f"  [{title}]: '{desc}'")

print("\n=== Sample 'decent' descriptions ===")
for title, desc in sample_descriptions["decent"]:
    print(f"  [{title}]: {desc[:200]}...")

print("\n=== Sample 'long' descriptions ===")
for title, desc in sample_descriptions["long"]:
    print(f"  [{title}]: {desc[:200]}...")

100%|██████████| 258585/258585 [00:09<00:00, 26182.49it/s]


Description length distribution:
  total          :  258,585 (100.0%)
  missing        :        0 (0.0%)
  empty_string   :   20,704 (8.0%)
  very_short     :      901 (0.3%)
  short          :    1,361 (0.5%)
  decent         :   46,686 (18.1%)
  long           :  188,933 (73.1%)

=== Sample 'very_short' descriptions (probably garbage) ===
  [A Kitty in the Lion's Den (Sweet Water, #3)]: 'Maverick's story.'
  [Dreamscape]: 'Out of print'
  [The Little Lame Prince]: 'victorian fairytale'

=== Sample 'decent' descriptions ===
  [The Passion (Dark Visions, #3)]: This is the final tale in the bestselling author L.J. Smith's romantic horrortrilogy. Now, Kaitlyn Fairchild and her friends must put the powerful crystalto the test--to stop an experiment that has tu...
  [Czaropis Tom II (Czaropis, #2)]: Mag Nicodemus staje do walki z demonami, polbogami i wlasna slaboscia w grze, ktorej stawka jest zycie wszystkich ludzi. Tym razem jednak nie jest sam. U boku bedzie mial Francesce, uzdolniona

In [5]:
def extract_book(book):
    """Pull just the fields we need, with proper type conversion."""
    # safe int conversion - returns None if missing/empty/invalid
    def safe_int(val):
        try:
            return int(val) if val not in ("", None) else None
        except (ValueError, TypeError):
            return None
    
    def safe_float(val):
        try:
            return float(val) if val not in ("", None) else None
        except (ValueError, TypeError):
            return None
    
    # author_ids - list of ids from the authors field
    author_ids = [a.get("author_id", "") for a in book.get("authors", []) if a.get("author_id")]
    
    # top shelves (excluding obvious noise like to-read, currently-reading, etc.)
    NOISE_SHELVES = {"to-read", "currently-reading", "owned", "favorites", "books-i-own",
                     "default", "read", "kindle", "ebook", "ebooks", "audiobook", "audiobooks",
                     "library", "wishlist", "tbr", "my-books", "owned-books", "have", "hardcover",
                     "paperback", "abandoned", "dnf", "did-not-finish"}
    shelves = [s["name"] for s in book.get("popular_shelves", []) 
               if s["name"].lower() not in NOISE_SHELVES][:10]  # top 10 non-noise
    
    return {
        "book_id": book.get("book_id", ""),
        "work_id": book.get("work_id", ""),
        "title": book.get("title", "").strip(),
        "description": book.get("description", "").strip(),
        "author_ids": author_ids,
        "shelves": shelves,
        "language_code": book.get("language_code", ""),
        "num_pages": safe_int(book.get("num_pages")),
        "publication_year": safe_int(book.get("publication_year")),
        "average_rating": safe_float(book.get("average_rating")),
        "ratings_count": safe_int(book.get("ratings_count")) or 0,
        "text_reviews_count": safe_int(book.get("text_reviews_count")) or 0,
        "is_ebook": book.get("is_ebook", False),
        "image_url": book.get("image_url", ""),
        "url": book.get("url", ""),
    }


def passes_filters(book):
    """v0 filters: English, has description of decent length, has some signal of existence."""
    if book.get("language_code", "") not in ("eng", "en-US", "en-GB", "en-CA", "en"):
        return False
    
    desc = book.get("description", "")
    if not desc or len(desc) < 100:
        return False
    
    if not book.get("title", "").strip():
        return False
    
    # filter out totally obscure books - need at least 5 ratings to show up
    try:
        ratings = int(book.get("ratings_count", "0"))
    except (ValueError, TypeError):
        ratings = 0
    if ratings < 5:
        return False
    
    return True


def process_genre_file(filepath, source_genre):
    """Stream through one genre file, filter, extract. Return list of dicts."""
    books = []
    with gzip.open(filepath, "rt", encoding="utf-8") as f:
        for line in tqdm(f, desc=f"  {filepath.name}"):
            book = json.loads(line)
            if not passes_filters(book):
                continue
            extracted = extract_book(book)
            extracted["source_genre"] = source_genre
            books.append(extracted)
    return books


# process each genre file that's downloaded
genre_files = {
    "fantasy_paranormal": DATA_DIR / "goodreads_books_fantasy_paranormal.json.gz",
    "mystery_thriller_crime": DATA_DIR / "goodreads_books_mystery_thriller_crime.json.gz",
    "history_biography": DATA_DIR / "goodreads_books_history_biography.json.gz",
}

all_books = []
for genre_name, filepath in genre_files.items():
    if not filepath.exists():
        print(f"SKIPPING {genre_name} - file not downloaded yet")
        continue
    print(f"\nProcessing {genre_name}...")
    books = process_genre_file(filepath, genre_name)
    print(f"  Kept {len(books):,} books after filtering")
    all_books.extend(books)

print(f"\n=== TOTAL: {len(all_books):,} books across all genres ===")


Processing fantasy_paranormal...


  goodreads_books_fantasy_paranormal.json.gz: 258585it [00:12, 21144.36it/s]


  Kept 128,179 books after filtering

Processing mystery_thriller_crime...


  goodreads_books_mystery_thriller_crime.json.gz: 219235it [00:09, 23214.10it/s]


  Kept 84,532 books after filtering

Processing history_biography...


  goodreads_books_history_biography.json.gz: 302935it [00:13, 21891.04it/s]

  Kept 85,827 books after filtering

=== TOTAL: 298,538 books across all genres ===


In [6]:
df = pd.DataFrame(all_books)
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nGenres breakdown:")
print(df["source_genre"].value_counts())

# books can appear in multiple genre files - check overlap
print(f"\nUnique book_ids: {df['book_id'].nunique():,}")
print(f"Total rows: {len(df):,}")
print(f"Duplicate rows: {len(df) - df['book_id'].nunique():,}")

# look at top books by ratings count (sanity check - should be recognizable titles)
print("\nTop 10 books by ratings_count:")
print(df.nlargest(10, "ratings_count")[["title", "source_genre", "ratings_count", "average_rating"]].to_string())

Shape: (298538, 16)

Columns: ['book_id', 'work_id', 'title', 'description', 'author_ids', 'shelves', 'language_code', 'num_pages', 'publication_year', 'average_rating', 'ratings_count', 'text_reviews_count', 'is_ebook', 'image_url', 'url', 'source_genre']

Genres breakdown:
source_genre
fantasy_paranormal        128179
history_biography          85827
mystery_thriller_crime     84532
Name: count, dtype: int64

Unique book_ids: 289,655
Total rows: 298,538
Duplicate rows: 8,883

Top 10 books by ratings_count:
                                                             title            source_genre  ratings_count  average_rating
86318     Harry Potter and the Sorcerer's Stone (Harry Potter, #1)      fantasy_paranormal        4765497            4.45
31783                                      Twilight (Twilight, #1)      fantasy_paranormal        3941381            3.57
264693                                       To Kill a Mockingbird       history_biography        3255518            4.2

In [7]:
# dedupe on book_id, keeping first occurrence (preserves whichever genre file processed it first)
# but combine shelves across duplicates for richer signal
df_deduped = df.drop_duplicates(subset="book_id", keep="first").reset_index(drop=True)
print(f"After dedup: {len(df_deduped):,} unique books")

# save to processed
output_path = PROCESSED_DIR / "books_v0.parquet"
df_deduped.to_parquet(output_path, index=False)
print(f"\nSaved to: {output_path}")
print(f"File size: {output_path.stat().st_size / (1024*1024):.1f} MB")

After dedup: 289,655 unique books

Saved to: ..\data\processed\books_v0.parquet
File size: 206.2 MB


In [8]:
from sentence_transformers import SentenceTransformer
import torch

# load the model onto GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
print(f"Model loaded. Output dim: {model.get_sentence_embedding_dimension()}")

# quick test - embed 3 sample descriptions and check it works
sample_descs = df_deduped["description"].head(3).tolist()
sample_embeddings = model.encode(sample_descs, convert_to_numpy=True, show_progress_bar=False)
print(f"\nTest embedding shape: {sample_embeddings.shape}")
print(f"First embedding (first 5 dims): {sample_embeddings[0][:5]}")

Using device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\aryas\Desktop\reccomendation-engine\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aryas\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

C:\Users\aryas\AppData\Local\Temp\ipykernel_26892\1950704051.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded. Output dim: {model.get_sentence_embedding_dimension()}")


Model loaded. Output dim: 384

Test embedding shape: (3, 384)
First embedding (first 5 dims): [-0.10912564 -0.07168486  0.05285531  0.02704851 -0.03416678]


In [9]:
import numpy as np

# reload deduped df if needed
df_deduped = pd.read_parquet(PROCESSED_DIR / "books_v0.parquet")

descriptions = df_deduped["description"].tolist()
print(f"Embedding {len(descriptions):,} descriptions...")

# batch_size 64 is a good starting point for 8GB VRAM
# if you OOM, drop to 32
embeddings = model.encode(
    descriptions,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,  # L2-normalize so cosine similarity = dot product
)

print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"Memory used: {embeddings.nbytes / (1024*1024):.1f} MB")

# save embeddings
embeddings_path = PROCESSED_DIR / "embeddings_minilm_v0.npy"
np.save(embeddings_path, embeddings)
print(f"Saved to: {embeddings_path}")

Embedding 289,655 descriptions...


Batches:   0%|          | 0/4526 [00:00<?, ?it/s]


Embeddings shape: (289655, 384)
Memory used: 424.3 MB
Saved to: ..\data\processed\embeddings_minilm_v0.npy


In [11]:
from rapidfuzz import fuzz, process

# load ratings
ratings_df = pd.read_csv("../data/my_ratings.csv")
# strip whitespace from string columns
for col in ["title", "author", "notes"]:
    if col in ratings_df.columns:
        ratings_df[col] = ratings_df[col].astype(str).str.strip()

print(f"Loaded {len(ratings_df)} ratings")
print(ratings_df[["title", "author", "rating"]].to_string())

# reload corpus
df_corpus = pd.read_parquet(PROCESSED_DIR / "books_v0.parquet")

# we need author names, not just IDs. Load the authors file.
print("\nLoading author names...")
author_id_to_name = {}
authors_file = DATA_DIR / "goodreads_book_authors.json.gz"
with gzip.open(authors_file, "rt", encoding="utf-8") as f:
    for line in f:
        a = json.loads(line)
        author_id_to_name[a["author_id"]] = a["name"]

def get_primary_author(author_ids):
    if author_ids is None or len(author_ids) == 0:
        return ""
    return author_id_to_name.get(author_ids[0], "")

df_corpus["primary_author"] = df_corpus["author_ids"].apply(get_primary_author)
print(f"Authors resolved: {(df_corpus['primary_author'] != '').sum():,} / {len(df_corpus):,}")

Loaded 28 ratings
                               title                      author  rating
0                        Dark Matter                Blake Crouch     3.5
1         The Eyes Are The Best Part                  Monika Kim     4.5
2      One Hundred Years of Solitude      Gabriel García Márquez     2.5
3           The Lovecraft Compendium              H.P. Lovecraft     2.0
4            The Palace of Illusions  Chitra Banerjee Divakaruni     4.5
5                      The Alchemist                Paulo Coelho     1.0
6                      The Poppy War                  R.F. Kuang     3.0
7                          Divergent               Veronica Roth     1.5
8              Thank You For Smoking         Christopher Buckley     4.0
9                         Yellowface                  R.F. Kuang     3.5
10              Kitchen Confidential            Anthony Bourdain     4.0
11           Interpreter of Maladies               Jhumpa Lahiri     3.0
12      The Storied Life of AJ Fi

In [ ]:
def find_best_match(query_title, query_author, corpus_df, title_threshold=80, author_threshold=70):
    """
    Find the best matching book in the corpus.
    Returns (book_id, matched_title, matched_author, score) or None.
    """
    # build search corpus of "title by author" strings for ranking
    # but we need book_ids back, so we use indices
    
    # first, narrow by title fuzzy match
    titles = corpus_df["title"].fillna("").tolist()
    
    # get top 20 title matches
    title_matches = process.extract(
        query_title, titles, scorer=fuzz.WRatio, limit=20
    )
    
    if not title_matches or title_matches[0][1] < title_threshold:
        return None
    
    # among top title matches, find one whose author also matches
    best = None
    best_score = 0
    for matched_title, title_score, idx in title_matches:
        candidate = corpus_df.iloc[idx]
        candidate_author = candidate["primary_author"]
        
        author_score = fuzz.WRatio(query_author, candidate_author) if query_author else 100
        
        # combined score - title weighted more heavily
        combined = title_score * 0.6 + author_score * 0.4
        
        if combined > best_score and author_score >= author_threshold:
            best_score = combined
            best = {
                "book_id": candidate["book_id"],
                "matched_title": matched_title,
                "matched_author": candidate_author,
                "title_score": title_score,
                "author_score": author_score,
                "combined_score": combined,
                "ratings_count": candidate["ratings_count"],
            }
    
    return best


# match all ratings
print("Matching your ratings to corpus...\n")
matched_ratings = []
unmatched = []

for _, row in ratings_df.iterrows():
    result = find_best_match(row["title"], row["author"], df_corpus)
    if result:
        result["my_title"] = row["title"]
        result["my_author"] = row["author"]
        result["rating"] = row["rating"]
        matched_ratings.append(result)
        print(f"✓ {row['title']:40s} -> {result['matched_title'][:50]:50s} (T:{result['title_score']:.0f} A:{result['author_score']:.0f})")
    else:
        unmatched.append({"title": row["title"], "author": row["author"], "rating": row["rating"]})
        print(f"✗ {row['title']:40s} -> NO MATCH")

print(f"\nMatched: {len(matched_ratings)} / {len(ratings_df)}")
print(f"Unmatched: {len(unmatched)}")

if unmatched:
    print("\nUnmatched books (not in your corpus):")
    for u in unmatched:
        print(f"  - {u['title']} by {u['author']}")